# HeartTwin coupled patient models — software validation
Run in a standard CPU Colab runtime. Pins the implementation and six components. Checks four-cavity spatial wall/circulation coupling, native patient calibration, raw image geometry, segmentation contracts and owner hospital boundaries with synthetic fixtures. Imaging extras are installed explicitly. The nnU-Net test uses a synthetic process contract, not a trained anatomical model. No patient/clinical qualification is implied.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, traceback, zipfile
from google.colab import files
SOURCE_COMMIT = 'c2aa34f99453713207d10dc26d11b1fe161a7815'
COMPONENTS = {'Virelion-CardiEP': 'f9bfcb78cdb857e5995ce244d8645f7370d6b3bb', 'Virelion-CardiAnatomy': 'e3cc3450c49b84334dd2da696f33a75f4f61557f', 'Virelion-CardiInfer': 'b4f0c3c30da5986c6a98dba6ceccf1e1de95c881', 'Virelion-CardiMech': '5dc14d886ccbda5a1398b863f9ae7821bbffa18b', 'Virelion-CardiFlow': '670d370635ba09b2908f7e6736299719e86fc36b', 'Virelion-ElectroTrace': '51adddec49efa182f9707524cb59a16b4be554d7'}
ROOT = Path('/content/HeartTwin_Patient_Coupled') / SOURCE_COMMIT[:12]
ROOT.mkdir(parents=True, exist_ok=True)
RESULTS = ROOT / 'results'; RESULTS.mkdir(exist_ok=True)
ENV = dict(os.environ, MPLBACKEND='Agg', PYTHONUNBUFFERED='1')

def run(args, name, cwd=None, timeout=1800):
    print('Starting:', name, flush=True)
    with (RESULTS / (name+'.log')).open('w') as log:
        completed = subprocess.run(list(map(str,args)), cwd=cwd, env=ENV, stdout=log, stderr=subprocess.STDOUT, timeout=timeout)
    if completed.returncode:
        print((RESULTS/(name+'.log')).read_text()[-12000:])
        raise RuntimeError(name+' failed with exit '+str(completed.returncode))

status={'status':'STARTED','scope':'Software and synthetic checks only; no patient accuracy or clinical qualification','source_commit':SOURCE_COMMIT,'components':COMPONENTS}
try:
    repo=ROOT/'source'
    if not repo.exists():
        run(['git','init',repo],'git-init')
        run(['git','-C',repo,'fetch','--depth','1','https://github.com/Virelion-Biotech/Virelion-HeartTwin.git',SOURCE_COMMIT],'git-fetch')
        run(['git','-C',repo,'checkout','--detach','FETCH_HEAD'],'git-checkout')
    observed=subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
    if observed != SOURCE_COMMIT: raise ValueError('Source commit mismatch')
    if subprocess.check_output(['git','-C',str(repo),'status','--porcelain','--untracked-files=no'],text=True).strip(): raise ValueError('Cached source contains modified tracked files')
    (RESULTS/'protocol.json').write_text(json.dumps(status,indent=2))
    venv=ROOT/'venv'
    if not venv.exists(): run([sys.executable,'-m','venv','--without-pip',venv],'venv')
    python=venv/'bin/python'
    ENV['PATH'] = str(python.parent) + os.pathsep + ENV['PATH']
    requirements=['pip==25.1.1','numpy==2.2.6','pydantic==2.12.3','PyYAML==6.0.3','pytest==8.4.2','jsonschema==4.25.1','scipy==1.16.3','ruff==0.14.1','nibabel==5.3.2','pydicom==3.0.1']
    run([sys.executable,'-m','pip','--python',python,'install',*requirements],'install-base')
    run([python,'-m','pip','install','-e',repo],'install-hearttwin')
    # Only synthetic fixtures are used; no hospital or private files are uploaded.
    for name, commit in COMPONENTS.items():
        run([python,'-m','pip','install','git+https://github.com/Virelion-Biotech/'+name+'.git@'+commit],'install-'+name)
    run([python,'-m','pip','check'],'pip-check')
    with (RESULTS/'environment.txt').open('w') as out: subprocess.run([python,'-m','pip','freeze'],stdout=out,check=True)
    run([python,repo/'scripts/validate_patient_lifecycle.py','--output',RESULTS/'checks','--components'],'validation',cwd=repo)
    print((RESULTS/'checks/patient-and-workbench-tests.log').read_text()[-3000:])
    status.update(status='SOFTWARE_PASSED')
except BaseException:
    status.update(status='FAILED', error=traceback.format_exc())
    print(status['error'])
finally:
    (RESULTS/'RUN_STATUS.json').write_text(json.dumps(status,indent=2))
    hashes={str(p.relative_to(RESULTS)):hashlib.sha256(p.read_bytes()).hexdigest() for p in RESULTS.rglob('*') if p.is_file() and p.name!='ARTIFACT_HASHES.json'}
    (RESULTS/'ARTIFACT_HASHES.json').write_text(json.dumps(hashes,indent=2))
    archive=ROOT/'HeartTwin_Patient_Coupled_Results.zip'
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in RESULTS.rglob('*'):
            if p.is_file(): z.write(p,str(p.relative_to(RESULTS)))
    print('STATUS:',status['status'])
    print('This run does not establish patient prediction accuracy or clinical qualification.')
    files.download(str(archive))
if status['status']!='SOFTWARE_PASSED': raise RuntimeError('Validation failed; retain the downloaded results ZIP')
